In [7]:
#!pip install selenium webdriver-manager
#pip install requests beautifulsoup4 pandas

### 1. Configuración de Rutas
Esta función prepara las direcciones de las carpetas 01_Raw y 02_Procesados.

In [8]:
import os

def configurar_rutas():
    # En Notebook, os.getcwd() apunta a la carpeta '00_Script'
    ruta_script = os.getcwd() 
    ruta_base = os.path.dirname(ruta_script)
    
    return {
        "raw": os.path.join(ruta_base, "01_Raw"),
        "procesados": os.path.join(ruta_base, "02_Procesados")
    }

### 2. Función: Extracción (Selenium)
Solo se encarga de entrar a la web y bajar el archivo .zip con todos los capítulos marcados.

In [9]:
from selenium import webdriver
from selenium.webdriver.edge.options import Options as EdgeOptions
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

import time
from selenium.common.exceptions import TimeoutException

def extraer_desde_web(anio, ruta_descarga):
    print(f"--- Iniciando descarga de {anio} ---")
    options = EdgeOptions()
    # Aseguramos que descargue en la carpeta 01_Raw
    options.add_experimental_option("prefs", {
        "download.default_directory": ruta_descarga,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    })
    
    driver = webdriver.Edge(options=options)
    wait = WebDriverWait(driver, 30)
    
    try:
        driver.get("https://servicio.mapa.gob.es/incendios/Search/Publico")
        time.sleep(5) # Tiempo de carga inicial que usabas antes

        # 1. Configurar año usando tu script de inyección directa
        print(f"Configurando filtro: Año {anio}...")
        driver.execute_script(f"document.querySelectorAll('input').forEach(i => {{ if(i.id.includes('txtNumAnio')) i.value = '{anio}' }});")

        # 2. Pulsar BUSCAR
        btn_buscar = wait.until(EC.element_to_be_clickable((By.ID, "btnBusqueda")))
        driver.execute_script("arguments[0].click();", btn_buscar)

        # 3. Esperar resultados y abrir exportación
        time.sleep(10) 
        btn_azul = wait.until(EC.element_to_be_clickable((By.ID, "idExportacionDatos")))
        driver.execute_script("arguments[0].click();", btn_azul)

        # 4. Configurar ventana modal (TODOS los capítulos)
        time.sleep(4) 
        script_modal = """
            const chkTodos = document.getElementById('xmlTODOS_id');
            const radioXml = document.getElementById('idTipoExportacionXML');
            if (radioXml && !radioXml.checked) radioXml.click();
            if (chkTodos && !chkTodos.checked) {
                chkTodos.click();
                chkTodos.dispatchEvent(new Event('change', { bubbles: true }));
            }
        """
        driver.execute_script(script_modal)

        # 5. Solicitar generación del ZIP
        btn_generar = wait.until(EC.presence_of_element_located((By.ID, "btnExportarDatosFullxmlCapitulosZip")))
        driver.execute_script("arguments[0].click();", btn_generar)

        # 6. Espera larga para el botón de descarga final
        print("Esperando a que el servidor genere el XML...")
        wait_largo = WebDriverWait(driver, 300) 
        btn_descargar_final = wait_largo.until(EC.element_to_be_clickable((By.ID, "btnDescargaXmlCapitulos")))
        
        driver.execute_script("arguments[0].click();", btn_descargar_final)
        time.sleep(25) # Tiempo para completar la descarga física en 01_Raw
        
    except Exception as e:
        print(f"❌ Error en el proceso web: {e}")
        raise 
    finally:
        driver.quit()

### 3. Función: Descompresión
Busca el ZIP más reciente en 01_Raw y saca el XML.

In [10]:
import zipfile
import glob

def descomprimir_archivo(ruta_raw):
    zips = glob.glob(os.path.join(ruta_raw, "*.zip"))
    if not zips:
        return None
    ultimo_zip = max(zips, key=os.path.getmtime)
    
    with zipfile.ZipFile(ultimo_zip, 'r') as zip_ref:
        zip_ref.extractall(ruta_raw)
        return os.path.join(ruta_raw, zip_ref.namelist()[0])

### 4. Función: Limpieza y Procesamiento
Transforma el XML en un CSV compatible con Power BI.

In [11]:
import xml.etree.ElementTree as ET
import pandas as pd

def procesar_datos(xml_path, anio, ruta_csv):
    print(f"Procesando XML de {anio}...")
    context = ET.iterparse(xml_path, events=('end',))
    data = []
    
    for event, elem in context:
        if elem.tag.endswith('pif') and len(list(elem)) > 2:
            fila = {sub.tag.split('}')[-1]: sub.text.strip() for sub in elem.iter() if sub.text and sub.text.strip()}
            data.append(fila)
            elem.clear()

    df = pd.DataFrame(data)
    
    # Asegurar IDs numéricos para las relaciones en Power BI
    cols_id = ['idpif', 'idcomunidad', 'idprovincia', 'idmunicipio']
    for col in cols_id:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    archivo_final = os.path.join(ruta_csv, f"hechos_incendios_{anio}.csv")
    df.to_csv(archivo_final, index=False, encoding='utf-8-sig')
    return archivo_final

### 5. Función Principal (Orquestador)
Esta es la función que "llama a todas" en orden.

In [12]:
def ejecutar_limpieza_incendios(lista_anios):
    rutas = configurar_rutas()
    
    for anio in lista_anios:
        # 1. Extraer
        extraer_desde_web(anio, rutas["raw"])
        
        # 2. Descomprimir
        xml_file = descomprimir_archivo(rutas["raw"])
        
        if xml_file:
            # 3. Procesar
            csv_generado = procesar_datos(xml_file, anio, rutas["procesados"])
            print(f"✅ Finalizado: {csv_generado}")
        else:
            print(f"❌ No se pudo encontrar el archivo para el año {anio}")

# PARA ARRANCAR TODO:
años_a_rescatar = list(range(2005, 2018)) # Esto genera [2005, 2006, ..., 2017]
ejecutar_limpieza_incendios(años_a_rescatar)

--- Iniciando descarga de 2005 ---


KeyboardInterrupt: 